<a href="https://colab.research.google.com/github/zubaeteee20210205109-droid/Thesis/blob/main/new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CELL 1: CHECK GPU
# ============================================================

import tensorflow as tf

print("=" * 60)
print("TensorFlow Version :", tf.__version__)
print("=" * 60)

print("Built with CUDA :", tf.test.is_built_with_cuda())

gpus = tf.config.list_physical_devices("GPU")

print("Available GPUs :", gpus)

if gpus:
    print("\nGPU DETECTED")

    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

    gpu_details = tf.config.experimental.get_device_details(gpus[0])
    print("GPU Name :", gpu_details.get("device_name"))

else:
    print("\nNO GPU DETECTED")
    print("TensorFlow is running on CPU.")

print("=" * 60)

In [ ]:
# ============================================================
# CELL 2: AUTOMATIC MULTI-RUN CONFIGURATION
# ============================================================

import os
import re
import random
import numpy as np
import tensorflow as tf

# Use the SAME experiment folder for all five runs
BASE_SAVE_DIR = "/content/drive/MyDrive/Thesis_Experiments"

EXPERIMENT_NAME = (
    "Arsenicosis_EfficientNetV2_Transformer"
)

EXPERIMENT_DIR = os.path.join(
    BASE_SAVE_DIR,
    EXPERIMENT_NAME
)

os.makedirs(EXPERIMENT_DIR, exist_ok=True)

# Automatically identify the next available run number
existing_runs = []

for folder in os.listdir(EXPERIMENT_DIR):
    match = re.fullmatch(r"Run_(\d+)", folder)

    if match and os.path.isdir(os.path.join(EXPERIMENT_DIR, folder)):
        existing_runs.append(int(match.group(1)))

RUN_NUMBER = max(existing_runs, default=0) + 1

# Automatically assign a different seed to each run
RUN_SEED = 101 * RUN_NUMBER

# Set random seeds
random.seed(RUN_SEED)
np.random.seed(RUN_SEED)
tf.keras.utils.set_random_seed(RUN_SEED)

# Enable deterministic TensorFlow operations where supported
try:
    tf.config.experimental.enable_op_determinism()
except Exception as e:
    print("Determinism configuration:", e)

print("=" * 60)
print("AUTOMATIC RUN CONFIGURATION")
print("=" * 60)

print("Experiment folder:", EXPERIMENT_DIR)
print("Selected run:", f"Run_{RUN_NUMBER}")
print("Random seed:", RUN_SEED)

print("\nExisting runs:", sorted(existing_runs))
print("New run will be saved separately.")
print("=" * 60)

In [ ]:
# ============================================================
# CELL 3: COPY DATASETS TO LOCAL SSD
# ============================================================

import os
import shutil

# Google Drive paths
ORIGINAL_SOURCE = "/content/drive/MyDrive/ArsenicSkinImage(BD)/Original"
AUGMENTED_SOURCE = "/content/drive/MyDrive/ArsenicSkinImage(BD)/Augmented"

# Local SSD paths
ORIGINAL_DESTINATION = "/content/Original"
AUGMENTED_DESTINATION = "/content/Augmented"

# Copy Original Dataset
if not os.path.exists(ORIGINAL_DESTINATION):

    print("Copying Original dataset to local SSD...")

    shutil.copytree(
        ORIGINAL_SOURCE,
        ORIGINAL_DESTINATION
    )

    print("Original dataset copied successfully!")

else:
    print("Original dataset already exists in local SSD.")

# Copy Augmented Dataset
if not os.path.exists(AUGMENTED_DESTINATION):

    print("\nCopying Augmented dataset to local SSD...")

    shutil.copytree(
        AUGMENTED_SOURCE,
        AUGMENTED_DESTINATION
    )

    print("Augmented dataset copied successfully!")

else:
    print("Augmented dataset already exists in local SSD.")

print("\nBoth datasets are ready!")

In [ ]:
# ============================================================
# CELL 4: CHECK DATASET FOLDERS
# ============================================================

print("ORIGINAL DATASET")

for folder in os.listdir(ORIGINAL_DESTINATION):

    folder_path = os.path.join(ORIGINAL_DESTINATION, folder)

    if os.path.isdir(folder_path):
        print(folder, ":", len(os.listdir(folder_path)), "files")


print("\nAUGMENTED DATASET")

for folder in os.listdir(AUGMENTED_DESTINATION):

    folder_path = os.path.join(AUGMENTED_DESTINATION, folder)

    if os.path.isdir(folder_path):
        print(folder, ":", len(os.listdir(folder_path)), "files")

In [ ]:
# ============================================================
# CELL 5: IMPORT REQUIRED LIBRARIES
# ============================================================

import os
import re
import pandas as pd
from pathlib import Path

print("Required libraries imported successfully!")

In [ ]:
# ============================================================
# CELL 6: SOURCE ID EXTRACTION
# ============================================================

def get_source_id(filename):

    stem = Path(filename).stem

    source_id = re.sub(
        r"_augmented.*$",
        "",
        stem,
        flags=re.IGNORECASE
    )

    return source_id.strip().lower()

In [ ]:
# ============================================================
# CELL 7: COLLECT ORIGINAL AND AUGMENTED IMAGE INFORMATION
# ============================================================

def collect_images(root_dir, dataset_type):

    records = []

    # Map actual folder names to standardized class names
    if dataset_type == "Original":

        class_mapping = {
            "infacted": "infected",
            "not_infacted": "not_infected"
        }

    else:

        class_mapping = {
            "infected": "infected",
            "not_infected": "not_infected"
        }

    for folder_name, class_name in class_mapping.items():

        class_dir = os.path.join(root_dir, folder_name)

        if not os.path.isdir(class_dir):
            print("WARNING: Missing folder:", class_dir)
            continue

        for filename in os.listdir(class_dir):

            file_path = os.path.join(class_dir, filename)

            if os.path.isfile(file_path) and filename.lower().endswith(
                (".jpg", ".jpeg", ".png", ".bmp")
            ):

                records.append({
                    "filename": filename,
                    "class_name": class_name,
                    "source_id": get_source_id(filename),
                    "file_path": file_path,
                    "dataset_type": dataset_type
                })

    return pd.DataFrame(
        records,
        columns=[
            "filename",
            "class_name",
            "source_id",
            "file_path",
            "dataset_type"
        ]
    )


# Collect datasets
original_df = collect_images(
    ORIGINAL_DESTINATION,
    "Original"
)

augmented_df = collect_images(
    AUGMENTED_DESTINATION,
    "Augmented"
)

print("=" * 60)
print("ORIGINAL TOTAL IMAGES :", len(original_df))
print("AUGMENTED TOTAL IMAGES:", len(augmented_df))
print("=" * 60)

print("\nOriginal class counts:")
print(original_df["class_name"].value_counts())

print("\nAugmented class counts:")
print(augmented_df["class_name"].value_counts())

In [ ]:
# ============================================================
# 8. SOURCE ID MATCHING AUDIT
# ============================================================

audit_results = []

for class_name in ["infected", "not_infected"]:

    # Get unique source IDs from Original dataset
    original_ids = set(
        original_df.loc[
            original_df["class_name"] == class_name,
            "source_id"
        ].dropna()
    )

    # Get unique source IDs from Augmented dataset
    augmented_ids = set(
        augmented_df.loc[
            augmented_df["class_name"] == class_name,
            "source_id"
        ].dropna()
    )

    # Compare source IDs
    matched_ids = original_ids.intersection(augmented_ids)

    original_only = original_ids.difference(augmented_ids)

    augmented_only = augmented_ids.difference(original_ids)

    # Store audit results
    audit_results.append({
        "class_name": class_name,
        "original_source_groups": len(original_ids),
        "augmented_source_groups": len(augmented_ids),
        "matched_source_groups": len(matched_ids),
        "original_only_groups": len(original_only),
        "augmented_only_groups": len(augmented_only)
    })

    print("\n" + "=" * 65)
    print("CLASS:", class_name)
    print("=" * 65)

    print("Original unique source IDs :", len(original_ids))
    print("Augmented unique source IDs:", len(augmented_ids))
    print("Matched source IDs         :", len(matched_ids))
    print("Original-only source IDs   :", len(original_only))
    print("Augmented-only source IDs  :", len(augmented_only))

    print("\nSample Original-only IDs:")
    print(sorted(original_only)[:15])

    print("\nSample Augmented-only IDs:")
    print(sorted(augmented_only)[:15])


# Display final audit summary
audit_df = pd.DataFrame(audit_results)

print("\n\n" + "=" * 65)
print("FINAL SOURCE ID MATCHING SUMMARY")
print("=" * 65)

display(audit_df)

In [ ]:

# ============================================================
# CELL 9: REUSE EXPERIMENT FOLDER AND SAVE AUDIT
# ============================================================

import os
import pandas as pd

# EXPERIMENT_DIR is already selected in Cell 1
os.makedirs(EXPERIMENT_DIR, exist_ok=True)

AUDIT_DIR = os.path.join(EXPERIMENT_DIR, "Dataset_Audit")
os.makedirs(AUDIT_DIR, exist_ok=True)

# Save audit summary
audit_df.to_csv(
    os.path.join(AUDIT_DIR, "Source_ID_Matching_Summary.csv"),
    index=False
)

# Save unmatched source IDs
unmatched_records = []

for class_name in ["infected", "not_infected"]:

    original_ids = set(
        original_df.loc[
            original_df["class_name"] == class_name,
            "source_id"
        ]
    )

    augmented_ids = set(
        augmented_df.loc[
            augmented_df["class_name"] == class_name,
            "source_id"
        ]
    )

    for source_id in sorted(original_ids - augmented_ids):

        rows = original_df[
            (original_df["class_name"] == class_name) &
            (original_df["source_id"] == source_id)
        ]

        for _, row in rows.iterrows():
            unmatched_records.append({
                "class_name": class_name,
                "source_id": source_id,
                "dataset": "Original-only",
                "filename": row["filename"],
                "file_path": row["file_path"]
            })

    for source_id in sorted(augmented_ids - original_ids):

        rows = augmented_df[
            (augmented_df["class_name"] == class_name) &
            (augmented_df["source_id"] == source_id)
        ]

        for _, row in rows.iterrows():
            unmatched_records.append({
                "class_name": class_name,
                "source_id": source_id,
                "dataset": "Augmented-only",
                "filename": row["filename"],
                "file_path": row["file_path"]
            })

unmatched_df = pd.DataFrame(unmatched_records)

unmatched_df.to_csv(
    os.path.join(AUDIT_DIR, "Unmatched_Source_IDs.csv"),
    index=False
)

print("Experiment folder:", EXPERIMENT_DIR)
print("Dataset audit saved successfully.")


In [ ]:
# ============================================================
# 10. INSPECT UNMATCHED NOT-INFECTED FILENAMES
# ============================================================

print("=" * 70)
print("ORIGINAL-ONLY NOT-INFECTED FILES")
print("=" * 70)

display(
    unmatched_df[
        (unmatched_df["class_name"] == "not_infected") &
        (unmatched_df["dataset"] == "Original-only")
    ][["source_id", "filename"]]
)

print("\n" + "=" * 70)
print("AUGMENTED-ONLY NOT-INFECTED FILES")
print("=" * 70)

display(
    unmatched_df[
        (unmatched_df["class_name"] == "not_infected") &
        (unmatched_df["dataset"] == "Augmented-only")
    ][["source_id", "filename"]].head(50)
)

print("\nTotal unmatched records:",
      len(unmatched_df))

In [ ]:
# ============================================================
# CELL 11: LEAKAGE-FREE GROUP-WISE DATASET SPLITTING
# Train      = Augmented images
# Validation = Original images
# Test       = Original images
# Split      = 70:15:15 by source ID
# ============================================================

from sklearn.model_selection import train_test_split
import pandas as pd
import os

RANDOM_STATE = 42

train_ids = set()
val_ids = set()
test_ids = set()

split_summary = []

for class_name in ["infected", "not_infected"]:

    # Only source IDs available in BOTH datasets
    original_ids = set(
        original_df.loc[
            original_df["class_name"] == class_name,
            "source_id"
        ]
    )

    augmented_ids = set(
        augmented_df.loc[
            augmented_df["class_name"] == class_name,
            "source_id"
        ]
    )

    matched_ids = sorted(original_ids.intersection(augmented_ids))

    # First split: 70% train, 30% temporary
    class_train_ids, temporary_ids = train_test_split(
        matched_ids,
        test_size=0.30,
        random_state=RANDOM_STATE
    )

    # Second split: 15% validation, 15% test
    class_val_ids, class_test_ids = train_test_split(
        temporary_ids,
        test_size=0.50,
        random_state=RANDOM_STATE
    )

    train_ids.update(class_train_ids)
    val_ids.update(class_val_ids)
    test_ids.update(class_test_ids)

    split_summary.append({
        "class_name": class_name,
        "matched_source_groups": len(matched_ids),
        "train_source_groups": len(class_train_ids),
        "validation_source_groups": len(class_val_ids),
        "test_source_groups": len(class_test_ids)
    })

# ============================================================
# FILTER IMAGES BY SOURCE ID
# ============================================================

# Training uses augmented images only
train_df = augmented_df[
    augmented_df["source_id"].isin(train_ids)
].copy()

# Validation uses original images only
val_df = original_df[
    original_df["source_id"].isin(val_ids)
].copy()

# Testing uses original images only
test_df = original_df[
    original_df["source_id"].isin(test_ids)
].copy()

# ============================================================
# VERIFY NO SOURCE ID LEAKAGE
# ============================================================

assert train_ids.isdisjoint(val_ids), "Leakage: Train and Validation overlap!"
assert train_ids.isdisjoint(test_ids), "Leakage: Train and Test overlap!"
assert val_ids.isdisjoint(test_ids), "Leakage: Validation and Test overlap!"

assert set(train_df["dataset_type"]) == {"Augmented"}
assert set(val_df["dataset_type"]) == {"Original"}
assert set(test_df["dataset_type"]) == {"Original"}

assert len(train_df) > 0
assert len(val_df) > 0
assert len(test_df) > 0

print("=" * 70)
print("SOURCE ID SPLIT VERIFICATION")
print("=" * 70)

print("Train source groups      :", len(train_ids))
print("Validation source groups :", len(val_ids))
print("Test source groups       :", len(test_ids))

print("\nNo source ID overlap detected.")
print("Training images   :", len(train_df))
print("Validation images :", len(val_df))
print("Testing images    :", len(test_df))

print("\nDataset source verification:")
print("Training  :", train_df["dataset_type"].unique())
print("Validation:", val_df["dataset_type"].unique())
print("Testing   :", test_df["dataset_type"].unique())

# ============================================================
# SPLIT SUMMARY
# ============================================================

split_summary_df = pd.DataFrame(split_summary)

display(split_summary_df)

print("\nClass-wise image counts:")

for split_name, dataframe in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df),
    ("TEST", test_df)
]:
    print(f"\n{split_name}")
    print(dataframe["class_name"].value_counts())

In [ ]:
# ============================================================
# CELL 12: SAVE SPLIT MANIFESTS AND SOURCE IDS
# ============================================================

SPLIT_DIR = os.path.join(EXPERIMENT_DIR, "Dataset_Split")
os.makedirs(SPLIT_DIR, exist_ok=True)

# Save image manifests
train_df.to_csv(
    os.path.join(SPLIT_DIR, "Train_Manifest.csv"),
    index=False
)

val_df.to_csv(
    os.path.join(SPLIT_DIR, "Validation_Manifest.csv"),
    index=False
)

test_df.to_csv(
    os.path.join(SPLIT_DIR, "Test_Manifest.csv"),
    index=False
)

# Save source ID lists
pd.DataFrame({"source_id": sorted(train_ids)}).to_csv(
    os.path.join(SPLIT_DIR, "Train_Source_IDs.csv"),
    index=False
)

pd.DataFrame({"source_id": sorted(val_ids)}).to_csv(
    os.path.join(SPLIT_DIR, "Validation_Source_IDs.csv"),
    index=False
)

pd.DataFrame({"source_id": sorted(test_ids)}).to_csv(
    os.path.join(SPLIT_DIR, "Test_Source_IDs.csv"),
    index=False
)

# Save split summary
split_summary_df.to_csv(
    os.path.join(SPLIT_DIR, "Split_Summary.csv"),
    index=False
)

print("=" * 70)
print("DATASET SPLIT FILES SAVED SUCCESSFULLY")
print("=" * 70)
print("Save location:", SPLIT_DIR)
print("Train, Validation and Test manifests saved.")
print("Source ID lists saved.")
print("Split summary saved.")

In [ ]:
# ============================================================
# CELL 13: FINAL DATASET INTEGRITY CHECK
# ============================================================

# Check duplicate image paths
assert train_df["file_path"].is_unique
assert val_df["file_path"].is_unique
assert test_df["file_path"].is_unique

# Check that all selected source IDs are represented
assert set(train_df["source_id"]) == train_ids
assert set(val_df["source_id"]) == val_ids
assert set(test_df["source_id"]) == test_ids

# Check class availability in all splits
for split_name, dataframe in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    counts = dataframe["class_name"].value_counts()

    assert "infected" in counts.index, f"{split_name}: infected missing"
    assert "not_infected" in counts.index, f"{split_name}: not_infected missing"

print("=" * 70)
print("ALL DATASET INTEGRITY CHECKS PASSED")
print("=" * 70)

print("No source ID leakage.")
print("No duplicate image paths within any split.")
print("Both classes are present in all splits.")
print("Training uses augmented images only.")
print("Validation and Test use original images only.")
print("\nDataset split is ready for model development.")

In [ ]:
# ============================================================
# CELL 14: CONFIGURATION AND TENSORFLOW DATASET PIPELINE
# ============================================================

import tensorflow as tf
import numpy as np
import pandas as pd
import os

# -----------------------------
# 1. Reproducibility
# -----------------------------
# Use the automatically selected run seed
SEED = RUN_SEED

tf.keras.utils.set_random_seed(SEED)

# -----------------------------
# 2. Model configuration
# -----------------------------
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 25

CLASS_NAMES = ['infected', 'not_infected']
NUM_CLASSES = len(CLASS_NAMES)

CLASS_TO_INDEX = {
    'infected': 0,
    'not_infected': 1
}

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))
print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Classes:", CLASS_TO_INDEX)

# -----------------------------
# 3. Verify split manifests
# -----------------------------
required_columns = {'file_path', 'class_name'}

for name, dataframe in [
    ('Train', train_df),
    ('Validation', val_df),
    ('Test', test_df)
]:
    missing = required_columns - set(dataframe.columns)

    if missing:
        raise ValueError(
            f"{name} manifest is missing columns: {missing}"
        )

    if not dataframe['file_path'].apply(os.path.isfile).all():
        raise FileNotFoundError(
            f"Some image files are missing from {name} manifest."
        )

    if not set(dataframe['class_name'].unique()).issubset(CLASS_NAMES):
        raise ValueError(f"Unexpected class found in {name} manifest.")

print("\nAll manifest paths and class labels verified.")

# -----------------------------
# 4. Image loading function
# -----------------------------
def load_image(image_path, class_name):

    image = tf.io.read_file(image_path)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)

    label = tf.one_hot(
        tf.cast(class_name, tf.int32),
        depth=NUM_CLASSES
    )

    return image, label


# -----------------------------
# 5. Dataset creation function
# -----------------------------
def create_dataset(dataframe, training=False):

    image_paths = dataframe['file_path'].astype(str).values

    labels = dataframe['class_name'].map(
        CLASS_TO_INDEX
    ).astype(np.int32).values

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, labels)
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


# -----------------------------
# 6. Create datasets
# -----------------------------
train_ds = create_dataset(train_df, training=True)
val_ds = create_dataset(val_df, training=False)
test_ds = create_dataset(test_df, training=False)

# -----------------------------
# 7. Verify batches
# -----------------------------
for images, labels in train_ds.take(1):

    print("\nTraining image shape:", images.shape)
    print("Training label shape:", labels.shape)

    print(
        "Pixel range:",
        float(tf.reduce_min(images)),
        "to",
        float(tf.reduce_max(images))
    )

print("\nDataset pipeline created successfully.")

print("Training batches:",
      tf.data.experimental.cardinality(train_ds).numpy())

print("Validation batches:",
      tf.data.experimental.cardinality(val_ds).numpy())

print("Testing batches:",
      tf.data.experimental.cardinality(test_ds).numpy())

In [ ]:
# ============================================================
# CELL 15: ORIGINAL EFFICIENTNETV2B0 + TRANSFORMER ARCHITECTURE
# ============================================================

import tensorflow as tf
from tensorflow.keras import layers, Model

# -----------------------------
# 1. Learnable positional embedding
# -----------------------------
class LearnablePositionEmbedding(layers.Layer):

    def build(self, input_shape):

        sequence_length = input_shape[1]
        embedding_dimension = input_shape[2]

        self.position_embeddings = self.add_weight(
            name="position_embeddings",
            shape=(1, sequence_length, embedding_dimension),
            initializer="random_normal",
            trainable=True
        )

        super().build(input_shape)

    def call(self, inputs):

        return inputs + self.position_embeddings


# -----------------------------
# 2. Transformer block
# -----------------------------
def transformer_block(inputs, embedding_dimension, block_number):

    # Multi-Head Self-Attention
    attention_output = layers.MultiHeadAttention(
        num_heads=4,
        key_dim=embedding_dimension // 4,
        name=f"transformer_{block_number}_mha"
    )(inputs, inputs)

    # Residual connection + Layer Normalization
    attention_output = layers.Add(
        name=f"transformer_{block_number}_attention_residual"
    )([inputs, attention_output])

    attention_output = layers.LayerNormalization(
        name=f"transformer_{block_number}_attention_norm"
    )(attention_output)

    # Feed-Forward Network
    ffn_output = layers.Dense(
        embedding_dimension * 2,
        activation='gelu',
        name=f"transformer_{block_number}_ffn_dense_1"
    )(attention_output)

    ffn_output = layers.Dense(
        embedding_dimension,
        name=f"transformer_{block_number}_ffn_dense_2"
    )(ffn_output)

    # Residual connection + Layer Normalization
    output = layers.Add(
        name=f"transformer_{block_number}_ffn_residual"
    )([attention_output, ffn_output])

    output = layers.LayerNormalization(
        name=f"transformer_{block_number}_ffn_norm"
    )(output)

    return output


# -----------------------------
# 3. EfficientNetV2B0 backbone
# -----------------------------
base_model = tf.keras.applications.EfficientNetV2B0(
    include_top=False,
    weights='imagenet',
    input_shape=(224, 224, 3),
    include_preprocessing=True
)

# Freeze all layers except the last 20
base_model.trainable = True

for layer in base_model.layers[:-20]:
    layer.trainable = False

for layer in base_model.layers[-20:]:
    layer.trainable = True

print("EfficientNetV2B0 total layers:", len(base_model.layers))
print("Trainable backbone layers:",
      sum(layer.trainable for layer in base_model.layers))


# -----------------------------
# 4. Model input and augmentation
# -----------------------------
inputs = layers.Input(
    shape=(224, 224, 3),
    name="input_image"
)

# Training-only augmentation
x = layers.RandomRotation(
    factor=40 / 360,
    name="random_rotation"
)(inputs)

x = layers.RandomZoom(
    height_factor=0.3,
    width_factor=0.3,
    name="random_zoom"
)(x)

x = layers.RandomFlip(
    mode="horizontal",
    name="random_horizontal_flip"
)(x)


# -----------------------------
# 5. EfficientNet feature extraction
# -----------------------------
x = base_model(x)

feature_height = x.shape[1]
feature_width = x.shape[2]
embedding_dimension = x.shape[-1]

print("\nFeature map shape:", x.shape)
print("Embedding dimension:", embedding_dimension)


# -----------------------------
# 6. Convert feature map into tokens
# -----------------------------
x = layers.Reshape(
    (-1, embedding_dimension),
    name="feature_map_to_tokens"
)(x)


# -----------------------------
# 7. Learnable positional embedding
# -----------------------------
x = LearnablePositionEmbedding(
    name="learnable_positional_embedding"
)(x)


# -----------------------------
# 8. Two Transformer blocks
# -----------------------------
x = transformer_block(
    x,
    embedding_dimension,
    block_number=1
)

x = transformer_block(
    x,
    embedding_dimension,
    block_number=2
)


# -----------------------------
# 9. Global Average Pooling
# -----------------------------
x = layers.GlobalAveragePooling1D(
    name="global_average_pooling"
)(x)


# -----------------------------
# 10. Classification head
# -----------------------------
x = layers.Dense(
    512,
    activation='relu',
    name="classifier_dense_512"
)(x)

x = layers.Dropout(
    0.4,
    name="classifier_dropout_0_4"
)(x)

x = layers.Dense(
    256,
    activation='relu',
    name="classifier_dense_256"
)(x)

x = layers.Dropout(
    0.3,
    name="classifier_dropout_0_3"
)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation='softmax',
    name="classification_output"
)(x)


# -----------------------------
# 11. Build complete model
# -----------------------------
model = Model(
    inputs=inputs,
    outputs=outputs,
    name="EfficientNetV2B0_TwoTransformer"
)


# -----------------------------
# 12. Compile model
# -----------------------------
optimizer = tf.keras.optimizers.AdamW(
    learning_rate=3e-4,
    weight_decay=1e-4
)

model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


# -----------------------------
# 13. Model summary
# -----------------------------
model.summary()

print("\nModel architecture created successfully.")
print("Transformer blocks: 2")
print("Attention heads per block: 4")
print("Classifier: 512 -> 256 -> 2")
print("Optimizer: AdamW")
print("Learning rate: 3e-4")
print("Weight decay: 1e-4")

In [ ]:

# ============================================================
# CELL 16: CLASS WEIGHTS AND TRAINING CALLBACKS
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

# ------------------------------------------------------------
# 1. CREATE UNIQUE RUN FOLDER
# ------------------------------------------------------------

RUN_DIR = os.path.join(
    EXPERIMENT_DIR,
    f"Run_{RUN_NUMBER}"
)

# Safety check: never overwrite an existing run
if os.path.exists(RUN_DIR):
    raise FileExistsError(
        f"{RUN_DIR} already exists. "
        "A new run folder must be selected."
    )

os.makedirs(RUN_DIR, exist_ok=False)

print("Current run:", f"Run_{RUN_NUMBER}")
print("Random seed:", RUN_SEED)
print("Run directory:", RUN_DIR)

# ------------------------------------------------------------
# 2. CALCULATE CLASS WEIGHTS
# ------------------------------------------------------------

y_train = train_df["class_name"].map(
    CLASS_TO_INDEX
).values

classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = {
    int(class_id): float(weight)
    for class_id, weight in zip(classes, weights)
}

print("\nClass distribution:")
print(train_df["class_name"].value_counts())

print("\nCalculated class weights:")
print(class_weights)

with open(
    os.path.join(RUN_DIR, "class_weights.json"),
    "w"
) as f:
    json.dump(class_weights, f, indent=4)

# ------------------------------------------------------------
# 3. MODEL CHECKPOINT
# ------------------------------------------------------------

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=os.path.join(RUN_DIR, "Best_Model.keras"),
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1
)

# ------------------------------------------------------------
# 4. EARLY STOPPING
# ------------------------------------------------------------

early_stopping_callback = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# ------------------------------------------------------------
# 5. LEARNING RATE REDUCTION
# ------------------------------------------------------------

reduce_lr_callback = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

# ------------------------------------------------------------
# 6. CSV TRAINING LOGGER
# ------------------------------------------------------------

csv_logger_callback = tf.keras.callbacks.CSVLogger(
    filename=os.path.join(RUN_DIR, "training_log.csv"),
    append=False
)

# ------------------------------------------------------------
# 7. CALLBACK LIST
# ------------------------------------------------------------

callbacks = [
    checkpoint_callback,
    early_stopping_callback,
    reduce_lr_callback,
    csv_logger_callback
]

print("\nCallbacks configured successfully.")
print("Checkpoint:", os.path.join(RUN_DIR, "Best_Model.keras"))
print("Training log:", os.path.join(RUN_DIR, "training_log.csv"))


In [ ]:
# ============================================================
# CELL 17: MODEL TRAINING
# ============================================================

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

# -----------------------------
# Save final training history
# -----------------------------
history_df = pd.DataFrame(history.history)

history_df.to_csv(
    os.path.join(RUN_DIR, "Training_History.csv"),
    index=False
)

# -----------------------------
# Display best epoch
# -----------------------------
best_epoch = int(
    np.argmax(history.history['val_accuracy']) + 1
)

best_val_accuracy = float(
    np.max(history.history['val_accuracy'])
)

print("\n" + "=" * 60)
print("TRAINING COMPLETED")
print("=" * 60)

print("Total epochs trained:", len(history.history['loss']))
print("Best epoch:", best_epoch)
print("Best validation accuracy:",
      f"{best_val_accuracy * 100:.2f}%")

print("\nSaved files:")
print("Best model:", os.path.join(RUN_DIR, "Best_Model.keras"))
print("Training log:", os.path.join(RUN_DIR, "training_log.csv"))
print("Training history:", os.path.join(RUN_DIR, "Training_History.csv"))

Epoch 1/25
130/168 ━━━━━━━━━━━━━━━━━━━━ 10:58 17s/step - accuracy: 0.7992 - loss: 0.5156

In [ ]:
# ============================================================
# CELL 18: LOAD BEST CHECKPOINT AND TEST EVALUATION
# ============================================================

import os
import numpy as np
import tensorflow as tf
import pandas as pd

# -----------------------------
# 1. Load best validation-accuracy model
# -----------------------------
BEST_MODEL_PATH = os.path.join(
    RUN_DIR,
    "Best_Model.keras"
)

best_model = tf.keras.models.load_model(
    BEST_MODEL_PATH,
    custom_objects={
        "LearnablePositionEmbedding": LearnablePositionEmbedding
    }
)

print("Best checkpoint loaded successfully.")

# -----------------------------
# 2. Evaluate on original test set
# -----------------------------
test_loss, test_accuracy = best_model.evaluate(
    test_ds,
    verbose=1
)

print("\n" + "=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)

print(f"Test Loss     : {test_loss:.6f}")
print(f"Test Accuracy : {test_accuracy * 100:.2f}%")

# -----------------------------
# 3. Generate predictions
# -----------------------------
y_prob = best_model.predict(
    test_ds,
    verbose=1
)

y_pred = np.argmax(y_prob, axis=1)

y_true = test_df['class_name'].map(
    CLASS_TO_INDEX
).values

# -----------------------------
# 4. Verify prediction dimensions
# -----------------------------
assert len(y_true) == len(y_pred) == len(y_prob)
assert y_prob.shape[1] == NUM_CLASSES

print("\nPrediction verification:")
print("Actual labels:", len(y_true))
print("Predicted labels:", len(y_pred))
print("Probability shape:", y_prob.shape)

# -----------------------------
# 5. Save predictions
# -----------------------------
predictions_df = pd.DataFrame({
    "filename": test_df["filename"].values,
    "file_path": test_df["file_path"].values,
    "source_id": test_df["source_id"].values,
    "actual_class": [
        CLASS_NAMES[i] for i in y_true
    ],
    "predicted_class": [
        CLASS_NAMES[i] for i in y_pred
    ],
    "infected_probability": y_prob[:, 0],
    "not_infected_probability": y_prob[:, 1],
    "correct_prediction": y_true == y_pred
})

predictions_df.to_csv(
    os.path.join(RUN_DIR, "Test_Predictions.csv"),
    index=False
)

print("\nTest predictions saved successfully.")

print("\nPrediction preview:")
display(predictions_df.head(10))

In [ ]:
# ============================================================
# CELL 19: EVALUATION METRICS AND CONFUSION MATRIX
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    precision_recall_curve,
    auc,
    matthews_corrcoef,
    cohen_kappa_score,
    balanced_accuracy_score
)

# ------------------------------------------------------------
# 1. Prepare true labels and predictions
# ------------------------------------------------------------

y_true = np.asarray(y_true).astype(int)
y_pred = np.asarray(y_pred).astype(int)
infected_prob = np.asarray(infected_prob).astype(float)

# Positive class: infected (class index 0)
y_true_binary = (y_true == 0).astype(int)

# ------------------------------------------------------------
# 2. Confusion Matrix
# ------------------------------------------------------------

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

TP = cm[0, 0]
FN = cm[0, 1]
FP = cm[1, 0]
TN = cm[1, 1]

# ------------------------------------------------------------
# 3. Calculate Evaluation Metrics
# ------------------------------------------------------------

accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true, y_pred,
    pos_label=0,
    zero_division=0
)

sensitivity = recall_score(
    y_true, y_pred,
    pos_label=0,
    zero_division=0
)

f1 = f1_score(
    y_true, y_pred,
    pos_label=0,
    zero_division=0
)

specificity = TN / (TN + FP) if (TN + FP) > 0 else 0

balanced_acc = balanced_accuracy_score(y_true, y_pred)

mcc = matthews_corrcoef(y_true, y_pred)

kappa = cohen_kappa_score(y_true, y_pred)

roc_auc = roc_auc_score(y_true_binary, infected_prob)

# Trapezoidal PR-AUC
precision_curve, recall_curve, _ = precision_recall_curve(
    y_true_binary,
    infected_prob
)

pr_auc = auc(recall_curve, precision_curve)

# ------------------------------------------------------------
# 4. Display Metrics
# ------------------------------------------------------------

metrics = {
    "Accuracy": accuracy,
    "Precision": precision,
    "Sensitivity": sensitivity,
    "Specificity": specificity,
    "F1_Score": f1,
    "Balanced_Accuracy": balanced_acc,
    "ROC_AUC": roc_auc,
    "PR_AUC": pr_auc,
    "MCC": mcc,
    "Cohen_Kappa": kappa,
    "TP": int(TP),
    "TN": int(TN),
    "FP": int(FP),
    "FN": int(FN)
}

print("=" * 60)
print("TEST SET EVALUATION RESULTS")
print("=" * 60)

for metric_name, metric_value in metrics.items():

    if isinstance(metric_value, (int, np.integer)):
        print(f"{metric_name:20s}: {metric_value}")

    else:
        print(f"{metric_name:20s}: {metric_value:.6f}")

print("=" * 60)

# ------------------------------------------------------------
# 5. Save Test Metrics CSV
# ------------------------------------------------------------

metrics_df = pd.DataFrame(
    list(metrics.items()),
    columns=["Metric", "Value"]
)

metrics_csv_path = os.path.join(
    RUN_DIR,
    "Test_Metrics.csv"
)

metrics_df.to_csv(
    metrics_csv_path,
    index=False
)

print("Saved:", metrics_csv_path)

# ------------------------------------------------------------
# 6. Save Confusion Matrix CSV
# ------------------------------------------------------------

confusion_df = pd.DataFrame(
    cm,
    index=["Actual_Infected", "Actual_Not_Infected"],
    columns=["Predicted_Infected", "Predicted_Not_Infected"]
)

confusion_csv_path = os.path.join(
    RUN_DIR,
    "Confusion_Matrix.csv"
)

confusion_df.to_csv(
    confusion_csv_path
)

print("Saved:", confusion_csv_path)

# ------------------------------------------------------------
# 7. Save Classification Report
# ------------------------------------------------------------

report = classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=["infected", "not_infected"],
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

report_csv_path = os.path.join(
    RUN_DIR,
    "Classification_Report.csv"
)

report_df.to_csv(report_csv_path)

print("Saved:", report_csv_path)

# ------------------------------------------------------------
# 8. Plot Confusion Matrix
# ------------------------------------------------------------

plt.figure(figsize=(7, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Infected", "Not Infected"],
    yticklabels=["Infected", "Not Infected"],
    annot_kws={"size": 14}
)

plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("Actual Label", fontsize=12)
plt.title("Confusion Matrix", fontsize=14)

plt.tight_layout()

cm_png_path = os.path.join(
    RUN_DIR,
    "Confusion_Matrix.png"
)

plt.savefig(
    cm_png_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

print("Saved:", cm_png_path)

# ------------------------------------------------------------
# 9. Save Publication-Quality PDF
# ------------------------------------------------------------

cm_pdf_path = os.path.join(
    RUN_DIR,
    "Confusion_Matrix.pdf"
)

plt.figure(figsize=(7, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Infected", "Not Infected"],
    yticklabels=["Infected", "Not Infected"],
    annot_kws={"size": 14}
)

plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("Actual Label", fontsize=12)
plt.title("Confusion Matrix", fontsize=14)

plt.tight_layout()

plt.savefig(
    cm_pdf_path,
    format="pdf",
    bbox_inches="tight"
)

plt.close()

print("Saved:", cm_pdf_path)

print("\nCell 19 completed successfully.")

In [ ]:
# ============================================================
# CELL 20: ROC CURVE AND PRECISION-RECALL CURVE
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_curve,
    precision_recall_curve,
    auc
)

# ------------------------------------------------------------
# 1. PREPARE BINARY LABELS AND PROBABILITIES
# ------------------------------------------------------------

# Positive class = infected (class index 0)

y_true_binary = (y_true == 0).astype(int)
infected_prob = y_prob[:, 0]

# ------------------------------------------------------------
# 2. ROC CURVE
# ------------------------------------------------------------

fpr, tpr, roc_thresholds = roc_curve(
    y_true_binary,
    infected_prob
)

roc_auc_value = auc(fpr, tpr)

roc_df = pd.DataFrame({
    "False_Positive_Rate": fpr,
    "True_Positive_Rate": tpr,
    "Threshold": roc_thresholds
})

roc_df.to_csv(
    os.path.join(RUN_DIR, "ROC_Curve_Data.csv"),
    index=False
)

# Plot ROC curve
plt.figure(figsize=(7, 6))

plt.plot(
    fpr,
    tpr,
    linewidth=2,
    label=f"Proposed Model (AUC = {roc_auc_value:.4f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1,
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(RUN_DIR, "ROC_Curve.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(RUN_DIR, "ROC_Curve.pdf"),
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 3. PRECISION-RECALL CURVE
# ------------------------------------------------------------

precision_values, recall_values, pr_thresholds = (
    precision_recall_curve(
        y_true_binary,
        infected_prob
    )
)

pr_auc_value = auc(
    recall_values,
    precision_values
)

pr_df = pd.DataFrame({
    "Precision": precision_values,
    "Recall": recall_values
})

pr_df.to_csv(
    os.path.join(RUN_DIR, "Precision_Recall_Curve_Data.csv"),
    index=False
)

# Plot Precision-Recall curve
plt.figure(figsize=(7, 6))

plt.plot(
    recall_values,
    precision_values,
    linewidth=2,
    label=f"Proposed Model (PR-AUC = {pr_auc_value:.4f})"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend(loc="lower left")
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(RUN_DIR, "Precision_Recall_Curve.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(RUN_DIR, "Precision_Recall_Curve.pdf"),
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 4. DISPLAY AND SAVE SUMMARY
# ------------------------------------------------------------

print("=" * 60)
print("CURVE SUMMARY")
print("=" * 60)

print(f"ROC-AUC : {roc_auc_value:.6f}")
print(f"PR-AUC  : {pr_auc_value:.6f}")

print("\nROC and Precision-Recall curve data saved successfully.")
print("Figures saved in PNG and PDF formats.")

print("\nExperiment directory:")
print(RUN_DIR)

In [ ]:
# ============================================================
# CELL 21: TRAINING AND VALIDATION ACCURACY/LOSS CURVES
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. LOAD TRAINING HISTORY
# ------------------------------------------------------------

HISTORY_PATH = os.path.join(
    RUN_DIR,
    "Training_History.csv"
)

history_df = pd.read_csv(HISTORY_PATH)

print("Training history loaded successfully.")
print("\nAvailable columns:")
print(history_df.columns.tolist())

display(history_df.head())

# ------------------------------------------------------------
# 2. IDENTIFY ACCURACY AND LOSS COLUMNS
# ------------------------------------------------------------

accuracy_col = "accuracy"
val_accuracy_col = "val_accuracy"
loss_col = "loss"
val_loss_col = "val_loss"

required_columns = [
    accuracy_col,
    val_accuracy_col,
    loss_col,
    val_loss_col
]

missing_columns = [
    col for col in required_columns
    if col not in history_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

epochs = range(1, len(history_df) + 1)

# ------------------------------------------------------------
# 3. TRAINING AND VALIDATION ACCURACY
# ------------------------------------------------------------

plt.figure(figsize=(8, 6))

plt.plot(
    epochs,
    history_df[accuracy_col],
    linewidth=2,
    label="Training Accuracy"
)

plt.plot(
    epochs,
    history_df[val_accuracy_col],
    linewidth=2,
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(RUN_DIR, "Accuracy_Curve.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(RUN_DIR, "Accuracy_Curve.pdf"),
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 4. TRAINING AND VALIDATION LOSS
# ------------------------------------------------------------

plt.figure(figsize=(8, 6))

plt.plot(
    epochs,
    history_df[loss_col],
    linewidth=2,
    label="Training Loss"
)

plt.plot(
    epochs,
    history_df[val_loss_col],
    linewidth=2,
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(RUN_DIR, "Loss_Curve.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(RUN_DIR, "Loss_Curve.pdf"),
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 5. SAVE TRAINING HISTORY COPY
# ------------------------------------------------------------

history_df.to_csv(
    os.path.join(RUN_DIR, "Training_History_Final.csv"),
    index=False
)

print("\nTraining history graphs saved successfully.")
print("Accuracy_Curve.png / Accuracy_Curve.pdf")
print("Loss_Curve.png / Loss_Curve.pdf")
print("Training_History_Final.csv")

In [ ]:
# ============================================================
# CELL 22: CORRECT AND INCORRECT PREDICTION VISUALIZATION
# ============================================================

import os
import matplotlib.pyplot as plt
from PIL import Image

# Separate correct and incorrect predictions
correct_df = predictions_df[
    predictions_df["correct_prediction"] == True
].head(5)

incorrect_df = predictions_df[
    predictions_df["correct_prediction"] == False
].head(5)

def show_prediction_samples(data, title, filename):
    if len(data) == 0:
        print(f"No samples available for: {title}")
        return

    fig, axes = plt.subplots(
        1, len(data),
        figsize=(4 * len(data), 5)
    )

    if len(data) == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, data.iterrows()):
        image = Image.open(row["file_path"]).convert("RGB")

        ax.imshow(image)
        ax.set_title(
            f"Actual: {row['actual_class']}\n"
            f"Predicted: {row['predicted_class']}",
            fontsize=10
        )
        ax.axis("off")

    fig.suptitle(title, fontsize=14)
    plt.tight_layout()

    plt.savefig(
        os.path.join(RUN_DIR, filename),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    plt.close(fig)

# Correct predictions
show_prediction_samples(
    correct_df,
    "Correctly Classified Test Images",
    "Correct_Predictions.png"
)

# Incorrect predictions
show_prediction_samples(
    incorrect_df,
    "Incorrectly Classified Test Images",
    "Incorrect_Predictions.png"
)

print("Prediction visualization completed.")

In [ ]:
# ============================================================
# CELL 23: FINAL EXPERIMENT SUMMARY
# ============================================================

import os
import json
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Define Summary Information
# ------------------------------------------------------------

summary = {
    "Experiment_Name": EXPERIMENT_NAME,
    "Run_Name": f"Run_{RUN_NUMBER}",
    "Run_Number": int(RUN_NUMBER),
    "Random_Seed": int(RUN_SEED),

    "Model": "EfficientNetV2B0 + Transformer",
    "Number_of_Transformer_Blocks": 2,

    "Dataset_Split": "Leakage-Free Group-Wise Split",
    "Random_State": 42,

    "Image_Size": "224x224",
    "Batch_Size": int(BATCH_SIZE),
    "Maximum_Epochs": int(EPOCHS),

    "Training_Samples": int(len(train_df)),
    "Validation_Samples": int(len(val_df)),
    "Testing_Samples": int(len(test_df))
}

# ------------------------------------------------------------
# 2. Add Test Metrics
# ------------------------------------------------------------

for _, row in metrics_df.iterrows():

    metric_name = str(row["Metric"])
    metric_value = row["Value"]

    if isinstance(metric_value, (np.integer, int)):
        summary[metric_name] = int(metric_value)

    else:
        summary[metric_name] = float(metric_value)

# ------------------------------------------------------------
# 3. Add Training Information
# ------------------------------------------------------------

history_csv_path = os.path.join(
    RUN_DIR,
    "Training_History.csv"
)

if os.path.exists(history_csv_path):

    history_df = pd.read_csv(history_csv_path)

    if "val_accuracy" in history_df.columns:

        best_epoch = int(
            history_df["val_accuracy"].idxmax() + 1
        )

        best_val_accuracy = float(
            history_df["val_accuracy"].max()
        )

        summary["Best_Validation_Accuracy"] = best_val_accuracy
        summary["Best_Validation_Accuracy_Epoch"] = best_epoch

    if "val_loss" in history_df.columns:

        best_val_loss_epoch = int(
            history_df["val_loss"].idxmin() + 1
        )

        best_val_loss = float(
            history_df["val_loss"].min()
        )

        summary["Best_Validation_Loss"] = best_val_loss
        summary["Best_Validation_Loss_Epoch"] = best_val_loss_epoch

    summary["Completed_Epochs"] = int(len(history_df))

# ------------------------------------------------------------
# 4. Save JSON Summary
# ------------------------------------------------------------

summary_json_path = os.path.join(
    RUN_DIR,
    "Final_Experiment_Summary.json"
)

with open(summary_json_path, "w") as file:

    json.dump(
        summary,
        file,
        indent=4
    )

print("Saved:", summary_json_path)

# ------------------------------------------------------------
# 5. Save CSV Summary
# ------------------------------------------------------------

summary_csv_path = os.path.join(
    RUN_DIR,
    "Final_Experiment_Summary.csv"
)

summary_df = pd.DataFrame(
    list(summary.items()),
    columns=["Parameter", "Value"]
)

summary_df.to_csv(
    summary_csv_path,
    index=False
)

print("Saved:", summary_csv_path)

# ------------------------------------------------------------
# 6. Display Final Summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL EXPERIMENT SUMMARY")
print("=" * 65)

for key, value in summary.items():

    print(f"{key:38s}: {value}")

print("=" * 65)

print("\nCell 23 completed successfully.")

In [ ]:
# ============================================================
# CELL 24: ZIP AND BACKUP CURRENT RUN
# ============================================================

import os
import zipfile

# ------------------------------------------------------------
# 1. Define ZIP File Path
# ------------------------------------------------------------

ZIP_FILE_PATH = os.path.join(
    EXPERIMENT_DIR,
    f"Run_{RUN_NUMBER}_Complete_Results.zip"
)

# ------------------------------------------------------------
# 2. Check Run Directory
# ------------------------------------------------------------

if not os.path.isdir(RUN_DIR):

    raise FileNotFoundError(
        f"Run directory not found: {RUN_DIR}"
    )

# ------------------------------------------------------------
# 3. Create ZIP Backup
# ------------------------------------------------------------

if os.path.exists(ZIP_FILE_PATH):

    os.remove(ZIP_FILE_PATH)

with zipfile.ZipFile(
    ZIP_FILE_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zipf:

    for root, dirs, files in os.walk(RUN_DIR):

        for file in files:

            file_path = os.path.join(root, file)

            relative_path = os.path.relpath(
                file_path,
                EXPERIMENT_DIR
            )

            zipf.write(
                file_path,
                arcname=relative_path
            )

# ------------------------------------------------------------
# 4. Verify ZIP File
# ------------------------------------------------------------

if os.path.exists(ZIP_FILE_PATH):

    zip_size_mb = os.path.getsize(
        ZIP_FILE_PATH
    ) / (1024 * 1024)

    with zipfile.ZipFile(ZIP_FILE_PATH, "r") as zipf:

        zip_contents = zipf.namelist()

        bad_file = zipf.testzip()

    if bad_file is not None:

        raise RuntimeError(
            f"ZIP verification failed: {bad_file}"
        )

    print("=" * 60)
    print("RUN BACKUP COMPLETED SUCCESSFULLY")
    print("=" * 60)

    print("Run Number :", RUN_NUMBER)
    print("Run Folder :", RUN_DIR)
    print("ZIP File   :", ZIP_FILE_PATH)
    print(f"ZIP Size   : {zip_size_mb:.2f} MB")
    print("Files      :", len(zip_contents))
    print("ZIP Status : Verified")

    print("=" * 60)

else:

    raise RuntimeError(
        "ZIP file was not created successfully."
    )

print("\nCell 24 completed successfully.")

In [ ]:
# ============================================================
# CELL 25: FINAL OUTPUT FILE VERIFICATION
# ============================================================

import os
import pandas as pd

# ------------------------------------------------------------
# 1. Define Expected Output Files
# ------------------------------------------------------------

expected_files = [
    "Best_Model.keras",
    "Class_Weights.json",
    "Training_History.csv",

    "Test_Predictions.csv",
    "Test_Metrics.csv",
    "Confusion_Matrix.csv",
    "Classification_Report.csv",
    "Confusion_Matrix.png",
    "Confusion_Matrix.pdf",

    "ROC_Curve.csv",
    "PR_Curve.csv",
    "ROC_Curve.png",
    "ROC_Curve.pdf",
    "PR_Curve.png",
    "PR_Curve.pdf",

    "Training_Accuracy_Loss_Curves.png",
    "Training_Accuracy_Loss_Curves.pdf",

    "Correct_Predictions",
    "Incorrect_Predictions",

    "Final_Experiment_Summary.json",
    "Final_Experiment_Summary.csv"
]

# ------------------------------------------------------------
# 2. Verify Each Output
# ------------------------------------------------------------

verification_results = []

for item in expected_files:

    item_path = os.path.join(RUN_DIR, item)

    if os.path.isfile(item_path):

        status = "FOUND"
        item_type = "File"
        size_mb = os.path.getsize(item_path) / (1024 * 1024)

    elif os.path.isdir(item_path):

        status = "FOUND"
        item_type = "Folder"
        size_mb = 0

    else:

        status = "MISSING"
        item_type = "Not Found"
        size_mb = 0

    verification_results.append({
        "Output_Name": item,
        "Type": item_type,
        "Status": status,
        "Size_MB": round(size_mb, 4)
    })

# ------------------------------------------------------------
# 3. Create Verification DataFrame
# ------------------------------------------------------------

verification_df = pd.DataFrame(
    verification_results
)

# ------------------------------------------------------------
# 4. Save Verification Report
# ------------------------------------------------------------

verification_csv_path = os.path.join(
    RUN_DIR,
    "Output_File_Verification.csv"
)

verification_df.to_csv(
    verification_csv_path,
    index=False
)

# ------------------------------------------------------------
# 5. Display Verification Results
# ------------------------------------------------------------

print("=" * 70)
print(f"OUTPUT FILE VERIFICATION - RUN {RUN_NUMBER}")
print("=" * 70)

print(verification_df.to_string(index=False))

# ------------------------------------------------------------
# 6. Verification Summary
# ------------------------------------------------------------

total_outputs = len(verification_df)

found_outputs = (
    verification_df["Status"] == "FOUND"
).sum()

missing_outputs = (
    verification_df["Status"] == "MISSING"
).sum()

print("\n" + "=" * 70)
print("VERIFICATION SUMMARY")
print("=" * 70)

print("Run Number       :", RUN_NUMBER)
print("Run Directory    :", RUN_DIR)
print("Total Outputs    :", total_outputs)
print("Found Outputs    :", found_outputs)
print("Missing Outputs  :", missing_outputs)

print("Verification CSV :", verification_csv_path)

if missing_outputs == 0:

    print("\nALL EXPECTED OUTPUTS FOUND.")

else:

    print("\nWARNING: SOME EXPECTED OUTPUTS ARE MISSING.")

    print("\nMissing Files:")

    print(
        verification_df.loc[
            verification_df["Status"] == "MISSING",
            "Output_Name"
        ].to_list()
    )

print("=" * 70)

print("\nCell 25 completed successfully.")